In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum

# Initialize Spark Session
spark = SparkSession.builder.appName("Ecommerce_Transformation").getOrCreate()

# List of all 12 tables[cite: 1]
tables = ["customers", "categories", "products", "departments", "employees", 
          "suppliers", "orders", "order_details", "payments", "product_suppliers", 
          "shippers", "shipments"]

dataframes = {}
raw_path = "s3://ecommerce-task28/raw/ecommerce/"

# 1. Read all 12 CSV files from S3[cite: 1]
for table in tables:
    df = spark.read.csv(f"{raw_path}{table}.csv", header=True, inferSchema=True)
    dataframes[table] = df
    
    print(f"--- Profiling {table.upper()} ---")
    
    # 2. Display schema[cite: 1]
    df.printSchema()
    
    # 3. Display 10 records[cite: 1]
    df.show(10)
    
    # 4. Calculate number of records[cite: 1]
    print(f"Record count for {table}: {df.count()}\n")

# Isolate customers and orders DataFrames
df_customers = dataframes["customers"]
df_orders = dataframes["orders"]

print("--- NULL Values in Customers Table ---")
# 5. Calculate NULL values for every column in customers[cite: 1]
null_counts = df_customers.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_customers.columns
])
null_counts.show()

print("--- Duplicate Customer IDs ---")
# 6. Find duplicate customerid values[cite: 1]
duplicate_customers = df_customers.groupBy("CustomerID").count().filter(col("count") > 1)
duplicate_customers.show()

print("--- Duplicate Order IDs ---")
# 7. Find duplicate orderid values[cite: 1]
duplicate_orders = df_orders.groupBy("OrderID").count().filter(col("count") > 1)
duplicate_orders.show()

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: e056c340-ce7d-4df2-ae9b-de121146099b
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session e056c340-ce7d-4df2-ae9b-de121146099b to get into ready status...
Session e056c340-ce7d-4df2-ae9b-de121146099b has been created.
--- Profiling CUSTOMERS ---
root
 |-- CustomerID: integer (nullable = true)
 |-- FirstName: string (nullable = true)
 |-- LastName: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Regis

In [2]:
from pyspark.sql.functions import col, lower, upper, trim
from pyspark.sql.types import StringType

# 8. Standardize all column names to lowercase[cite: 1]
# 9. Remove leading and trailing spaces from all string columns[cite: 1]
for table_name, df in dataframes.items():
    # Lowercase column names
    for c in df.columns:
        df = df.withColumnRenamed(c, c.lower())
    
    # Trim string columns
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, trim(col(c)))
        
    dataframes[table_name] = df

# Extract the main tables needed for specific transformations
df_customers = dataframes["customers"]
df_orders = dataframes["orders"]
df_products = dataframes["products"]

# 10. Convert customer emails to lowercase[cite: 1]
df_customers = df_customers.withColumn("email", lower(col("email")))

# 11. Convert order statuses to uppercase[cite: 1]
df_orders = df_orders.withColumn("status", upper(col("status")))

# 12. Remove duplicate customers based on customerid[cite: 1]
df_customers = df_customers.dropDuplicates(["customerid"])

# 13. Remove duplicate orders based on orderid[cite: 1]
df_orders = df_orders.dropDuplicates(["orderid"])

# 14. Remove records where the primary key is NULL[cite: 1]
df_customers = df_customers.dropna(subset=["customerid"])
df_orders = df_orders.dropna(subset=["orderid"])
df_products = df_products.dropna(subset=["productid"])
# (Note: Apply .dropna() to the PKs of other tables as needed)

# Reassign the cleaned DataFrames back to the dictionary
dataframes["customers"] = df_customers
dataframes["orders"] = df_orders
dataframes["products"] = df_products

print("--- Transformations 8 through 14 Applied ---")

# ==============================================================
# Finding Anomalies (Outputs directly beneath the cell)
# ==============================================================

# 15. Find customers with invalid email formats[cite: 1]
print("\n15. Customers with invalid emails:")
email_pattern = "^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$"
invalid_emails = df_customers.filter(~col("email").rlike(email_pattern))
invalid_emails.show()

# 16. Find orders where the status is not PENDING, SHIPPED, DELIVERED, or CANCELLED[cite: 1]
print("16. Orders with invalid statuses:")
valid_statuses = ["PENDING", "SHIPPED", "DELIVERED", "CANCELLED"]
invalid_statuses = df_orders.filter(~col("status").isin(valid_statuses))
invalid_statuses.show()

# 17. Find products with NULL prices[cite: 1]
print("17. Products with NULL prices:")
null_prices = df_products.filter(col("price").isNull())
null_prices.show()

# 18. Find products with prices less than or equal to zero[cite: 1]
print("18. Products with prices <= 0:")
invalid_prices = df_products.filter(col("price") <= 0)
invalid_prices.show()

--- Transformations 8 through 14 Applied ---

15. Customers with invalid emails:
+----------+---------+--------+-----+----+-------+----------------+
|customerid|firstname|lastname|email|city|country|registrationdate|
+----------+---------+--------+-----+----+-------+----------------+
+----------+---------+--------+-----+----+-------+----------------+

16. Orders with invalid statuses:
+-------+----------+-------------------+----------+
|orderid|customerid|          orderdate|    status|
+-------+----------+-------------------+----------+
|     12|      2908|2024-12-25 00:00:00|PROCESSING|
|     38|      2252|2025-08-15 00:00:00|PROCESSING|
|     69|       257|2024-11-03 00:00:00|PROCESSING|
|     77|      1862|2024-08-07 00:00:00|PROCESSING|
|     93|      6380|2025-09-29 00:00:00|PROCESSING|
|     94|      2361|2025-11-10 00:00:00|PROCESSING|
|    138|      9458|2025-08-18 00:00:00|PROCESSING|
|    180|      8180|2025-05-13 00:00:00|PROCESSING|
|    193|      9710|2025-11-18 00:00:00|

In [3]:
from pyspark.sql.types import LongType, DecimalType, IntegerType
from pyspark.sql.functions import to_date, year, month, quarter, dayofmonth, dayofweek

# 19. Convert all ID columns to long[cite: 1]
for table_name, df in dataframes.items():
    for col_name in df.columns:
        if col_name.endswith("id"):
            df = df.withColumn(col_name, col(col_name).cast(LongType()))
    dataframes[table_name] = df

# Extract specific DataFrames for targeted transformations
df_products = dataframes["products"]
df_order_details = dataframes["order_details"]
df_orders = dataframes["orders"]
df_payments = dataframes["payments"]

# 20. Convert prices to decimal(12,2)[cite: 1]
df_products = df_products.withColumn("price", col("price").cast(DecimalType(12, 2)))
df_order_details = df_order_details.withColumn("unitprice", col("unitprice").cast(DecimalType(12, 2)))

# 21. Convert order dates to Spark date[cite: 1]
df_orders = df_orders.withColumn("orderdate", to_date(col("orderdate")))

# 22. Convert payment amounts to decimal[cite: 1]
df_payments = df_payments.withColumn("amount", col("amount").cast(DecimalType(12, 2)))

# 23. Convert quantities to integer[cite: 1]
df_order_details = df_order_details.withColumn("quantity", col("quantity").cast(IntegerType()))

# 24. Extract date columns from orderdate[cite: 1]
df_orders = (df_orders
             .withColumn("year", year(col("orderdate")))
             .withColumn("month", month(col("orderdate")))
             .withColumn("quarter", quarter(col("orderdate")))
             .withColumn("day", dayofmonth(col("orderdate")))
             .withColumn("day_of_week", dayofweek(col("orderdate"))))

# Reassign the updated DataFrames back to the dictionary
dataframes["products"] = df_products
dataframes["order_details"] = df_order_details
dataframes["orders"] = df_orders
dataframes["payments"] = df_payments

print("--- Part 3: Data Type Transformations Applied ---")
# Verify the schema changes on the orders table
df_orders.printSchema()
df_orders.show(5)

--- Part 3: Data Type Transformations Applied ---
root
 |-- orderid: long (nullable = true)
 |-- customerid: long (nullable = true)
 |-- orderdate: date (nullable = true)
 |-- status: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- quarter: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)

+-------+----------+----------+----------+----+-----+-------+---+-----------+
|orderid|customerid| orderdate|    status|year|month|quarter|day|day_of_week|
+-------+----------+----------+----------+----+-----+-------+---+-----------+
|      1|      8870|2024-05-03| CANCELLED|2024|    5|      2|  3|          6|
|      3|      8030|2026-01-14| DELIVERED|2026|    1|      1| 14|          4|
|      5|      4623|2025-05-26| DELIVERED|2025|    5|      2| 26|          2|
|     10|      9903|2025-08-15| CANCELLED|2025|    8|      3| 15|          6|
|     12|      2908|2024-12-25|PROCESSING|2024|  

In [4]:
from pyspark.sql.functions import col, concat_ws, when, sum as spark_sum, avg as spark_avg

# 25. In order_details, create: total_amount = quantity * unitprice[cite: 1]
df_order_details = dataframes["order_details"].withColumn(
    "total_amount", col("quantity") * col("unitprice")
)

# 26. Create full_name in the customers table[cite: 1]
df_customers = dataframes["customers"].withColumn(
    "full_name", concat_ws(" ", col("firstname"), col("lastname"))
)

# 29. Calculate the total amount for every order[cite: 1]
# 30. Calculate the total quantity purchased for every order[cite: 1]
df_order_summary = df_order_details.groupBy("orderid").agg(
    spark_sum("total_amount").alias("order_total"),
    spark_sum("quantity").alias("total_order_quantity")
)

# 27. Create a column called order_value_category[cite: 1]
df_order_summary = df_order_summary.withColumn(
    "order_value_category",
    when(col("order_total") >= 1000, "HIGH")
    .when((col("order_total") >= 500) & (col("order_total") < 1000), "MEDIUM")
    .otherwise("LOW")
)

# 28. Create a column called customer_segment based on total customer sales[cite: 1]
# First, join orders with the summary to link customers to their order totals
df_customer_sales = dataframes["orders"].join(df_order_summary, "orderid") \
    .groupBy("customerid").agg(spark_sum("order_total").alias("total_customer_sales"))

# Apply segmentation rules (Assumed thresholds since none were provided)
df_customer_sales = df_customer_sales.withColumn(
    "customer_segment",
    when(col("total_customer_sales") >= 10000, "VIP")
    .when((col("total_customer_sales") >= 5000) & (col("total_customer_sales") < 10000), "PREMIUM")
    .when((col("total_customer_sales") >= 1000) & (col("total_customer_sales") < 5000), "REGULAR")
    .otherwise("LOW_VALUE")
)

# Join the segments back to the main customers DataFrame
df_customers = df_customers.join(df_customer_sales, "customerid", "left")

# 31. Calculate the average product price[cite: 1]
avg_price_df = dataframes["products"].select(spark_avg("price").alias("average_product_price"))
print("31. Average Product Price:")
avg_price_df.show()

# Update the main dictionary with the new DataFrames
dataframes["order_details"] = df_order_details
dataframes["customers"] = df_customers
# Saving order_summary as a new dataframe for future joins if needed
dataframes["order_summary"] = df_order_summary

print("--- Part 4: Business Transformations Applied ---")
df_customers.select("customerid", "full_name", "total_customer_sales", "customer_segment").show(5)
df_order_summary.show(5)

31. Average Product Price:
+---------------------+
|average_product_price|
+---------------------+
|          1530.095400|
+---------------------+

--- Part 4: Business Transformations Applied ---
+----------+---------------+--------------------+----------------+
|customerid|      full_name|total_customer_sales|customer_segment|
+----------+---------------+--------------------+----------------+
|         3|    Jill Rhodes|            34827.09|             VIP|
|         4|Patricia Miller|             3557.25|         REGULAR|
|         6| Jeffery Wagner|            39733.00|             VIP|
|         2|  Joshua Walker|           123346.99|             VIP|
|         5| Robert Johnson|           209374.88|             VIP|
+----------+---------------+--------------------+----------------+
only showing top 5 rows

+-------+-----------+--------------------+--------------------+
|orderid|order_total|total_order_quantity|order_value_category|
+-------+-----------+--------------------+-----

In [5]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, row_number, rank, desc, sum as spark_sum

# Extract necessary DataFrames
df_orders = dataframes["orders"]
df_customers = dataframes["customers"]
df_products = dataframes["products"]
df_shipments = dataframes["shipments"]
df_order_details = dataframes["order_details"]

# 32. Find the latest order for every customer[cite: 1]
window_latest_order = Window.partitionBy("customerid").orderBy(col("orderdate").desc())
latest_orders = df_orders.withColumn("rn", row_number().over(window_latest_order)) \
    .filter(col("rn") == 1) \
    .select("customerid", "orderid", "status", "orderdate")

print("32. Latest order for every customer:")
latest_orders.show(5)

# 33. Find the first order for every customer[cite: 1]
window_first_order = Window.partitionBy("customerid").orderBy(col("orderdate").asc())
first_orders = df_orders.withColumn("rn", row_number().over(window_first_order)) \
    .filter(col("rn") == 1) \
    .select("customerid", "orderid", "status", "orderdate")

print("33. First order for every customer:")
first_orders.show(5)

# 34. Rank customers by total sales[cite: 1]
window_sales_rank = Window.orderBy(col("total_customer_sales").desc())
ranked_customers = df_customers.filter(col("total_customer_sales").isNotNull()) \
    .withColumn("sales_rank", rank().over(window_sales_rank)) \
    .select("customerid", "full_name", "total_customer_sales", "sales_rank")

print("34. Customers ranked by total sales:")
ranked_customers.show(5)

# 35. Find the top 3 products in every category (Assuming 'top' means by total sales)[cite: 1]
# First, calculate total sales per product
product_sales_df = df_order_details.groupBy("productid").agg(spark_sum("total_amount").alias("product_sales"))
df_products_with_sales = df_products.join(product_sales_df, "productid", "left")

window_top_products = Window.partitionBy("categoryid").orderBy(col("product_sales").desc())
top_3_products_per_category = df_products_with_sales.withColumn("rn", row_number().over(window_top_products)) \
    .filter(col("rn") <= 3) \
    .select("categoryid", "productid", "productname", "product_sales", "rn")

print("35. Top 3 products in every category (by sales):")
top_3_products_per_category.show(5)

# 36. Find the most expensive product in every category[cite: 1]
window_expensive_product = Window.partitionBy("categoryid").orderBy(col("price").desc())
most_expensive_products = df_products.withColumn("rn", row_number().over(window_expensive_product)) \
    .filter(col("rn") == 1) \
    .select("categoryid", "productid", "productname", "price")

print("36. Most expensive product in every category:")
most_expensive_products.show(5)

# 37. Find the latest shipment for every order[cite: 1]
window_latest_shipment = Window.partitionBy("orderid").orderBy(col("shipdate").desc())
latest_shipments = df_shipments.withColumn("rn", row_number().over(window_latest_shipment)) \
    .filter(col("rn") == 1) \
    .drop("rn")

print("37. Latest shipment for every order:")
latest_shipments.show(5)

32. Latest order for every customer:
+----------+-------+----------+----------+
|customerid|orderid|    status| orderdate|
+----------+-------+----------+----------+
|         3|  17011|   PENDING|2026-04-01|
|         4|  21695|PROCESSING|2025-10-15|
|         6|  19312|   PENDING|2026-07-03|
|         9|   2861| DELIVERED|2025-11-29|
|        11|  33028|PROCESSING|2026-01-18|
+----------+-------+----------+----------+
only showing top 5 rows

33. First order for every customer:
+----------+-------+----------+----------+
|customerid|orderid|    status| orderdate|
+----------+-------+----------+----------+
|         3|  12675| CANCELLED|2024-01-08|
|         4|  21695|PROCESSING|2025-10-15|
|         6|  30917|   SHIPPED|2024-03-09|
|         9|  30999| DELIVERED|2024-02-28|
|        11|  26574| CANCELLED|2025-01-20|
+----------+-------+----------+----------+
only showing top 5 rows

34. Customers ranked by total sales:
+----------+---------------+--------------------+----------+
|cust

In [6]:
# 38. Join orders + customers and select specific columns[cite: 1]
df_orders_customers = dataframes["orders"].join(dataframes["customers"], "customerid") \
    .withColumnRenamed("full_name", "customer_name") \
    .select("orderid", "customerid", "customer_name", "city", "country", "orderdate", "status")

print("38. Orders + Customers (First 5 rows):")
df_orders_customers.show(5)

# 39. Join orders + order_details + products to create a sales dataset[cite: 1]
df_sales_dataset = dataframes["orders"] \
    .join(dataframes["order_details"], "orderid") \
    .join(dataframes["products"], "productid")

print("39. Sales Dataset (First 5 rows):")
df_sales_dataset.show(5)

# 40. Join products + categories[cite: 1]
df_products_categories = dataframes["products"].join(dataframes["categories"], "categoryid")

# 41. Join products + product_suppliers + suppliers[cite: 1]
df_products_suppliers = dataframes["products"] \
    .join(dataframes["product_suppliers"], "productid") \
    .join(dataframes["suppliers"], "supplierid")

# 42. Join orders with payments[cite: 1]
df_orders_payments = dataframes["orders"].join(dataframes["payments"], "orderid")

# 43. Join orders with shipments and shippers[cite: 1]
df_orders_shipments_shippers = dataframes["orders"] \
    .join(dataframes["shipments"], "orderid") \
    .join(dataframes["shippers"], "shipperid")

# Add the combined datasets to your dataframes dictionary for easy access later
dataframes["sales_dataset"] = df_sales_dataset
dataframes["orders_customers"] = df_orders_customers

print("--- Part 6: Joins Completed ---")

38. Orders + Customers (First 5 rows):
+-------+----------+----------------+-----+-------+----------+----------+
|orderid|customerid|   customer_name| city|country| orderdate|    status|
+-------+----------+----------------+-----+-------+----------+----------+
|  34482|         1|Danielle Johnson| Giza|  Egypt|2026-03-12|   SHIPPED|
|   5073|         1|Danielle Johnson| Giza|  Egypt|2024-01-01|   SHIPPED|
|  35385|         1|Danielle Johnson| Giza|  Egypt|2026-08-14| CANCELLED|
|  37134|         1|Danielle Johnson| Giza|  Egypt|2024-03-23|PROCESSING|
|  36443|         2|   Joshua Walker|Dubai| Jordan|2026-05-15|   SHIPPED|
+-------+----------+----------------+-----+-------+----------+----------+
only showing top 5 rows

39. Sales Dataset (First 5 rows):
+---------+-------+----------+----------+----------+----+-----+-------+---+-----------+-------------+--------+---------+--------+------------+----------+------------------+------+-------+-------+-----+
|productid|orderid|customerid| ord

In [7]:
from pyspark.sql.functions import col, sum as spark_sum, count, avg as spark_avg

# Extract the necessary DataFrames for aggregations
df_sales = dataframes["sales_dataset"] # Contains orders + order_details + products[cite: 1]
df_customers = dataframes["customers"]
df_orders = dataframes["orders"]
df_categories = dataframes["categories"]

# 44. Calculate total sales[cite: 1]
total_sales = df_sales.select(spark_sum("total_amount").alias("total_sales"))
print("44. Total Sales:")
total_sales.show()

# 45. Calculate the total number of orders[cite: 1]
total_orders = df_orders.select(count("orderid").alias("total_orders"))
print("45. Total Number of Orders:")
total_orders.show()

# 46. Calculate the total quantity sold[cite: 1]
total_quantity = df_sales.select(spark_sum("quantity").alias("total_quantity_sold"))
print("46. Total Quantity Sold:")
total_quantity.show()

# 47. Calculate the average order value[cite: 1]
# We use the order_summary created in Part 4 which aggregated total_amount per orderid
avg_order_value = dataframes["order_summary"].select(spark_avg("order_total").alias("average_order_value"))
print("47. Average Order Value:")
avg_order_value.show()

# 48. Calculate total sales by customer[cite: 1]
sales_by_customer = df_sales.groupBy("customerid").agg(spark_sum("total_amount").alias("total_sales"))

# 49. Calculate total sales by product[cite: 1]
sales_by_product = df_sales.groupBy("productid", "productname").agg(spark_sum("total_amount").alias("total_sales"))

# 50. Calculate total sales by category[cite: 1]
sales_by_category = df_sales.join(df_categories, "categoryid") \
    .groupBy("categoryid", "categoryname").agg(spark_sum("total_amount").alias("total_sales"))

# Join sales with customers to get geographical data for 51 and 52
df_sales_geography = df_sales.join(df_customers, "customerid")

# 51. Calculate total sales by country[cite: 1]
sales_by_country = df_sales_geography.groupBy("country").agg(spark_sum("total_amount").alias("total_sales"))

# 52. Calculate total sales by city[cite: 1]
sales_by_city = df_sales_geography.groupBy("city").agg(spark_sum("total_amount").alias("total_sales"))

# 53. Calculate monthly sales[cite: 1]
# Assuming we group by both year and month to avoid mixing months from different years
monthly_sales = df_sales.groupBy("year", "month").agg(spark_sum("total_amount").alias("total_sales")).orderBy("year", "month")

# 54. Calculate yearly sales[cite: 1]
yearly_sales = df_sales.groupBy("year").agg(spark_sum("total_amount").alias("total_sales")).orderBy("year")

# 55. Calculate sales by order status[cite: 1]
sales_by_status = df_sales.groupBy("status").agg(spark_sum("total_amount").alias("total_sales"))

# 56. Calculate the number of orders per customer[cite: 1]
orders_per_customer = df_orders.groupBy("customerid").agg(count("orderid").alias("order_count"))

# Save aggregations into the dictionary for the final output step later
dataframes["customer_sales"] = sales_by_customer
dataframes["product_sales"] = sales_by_product
dataframes["category_sales"] = sales_by_category
dataframes["monthly_sales"] = monthly_sales

print("--- Part 7: Aggregations Completed ---")
print("Monthly Sales Sample:")
monthly_sales.show(5)

44. Total Sales:
+------------+
| total_sales|
+------------+
|840507631.75|
+------------+

45. Total Number of Orders:
+------------+
|total_orders|
+------------+
|       50000|
+------------+

46. Total Quantity Sold:
+-------------------+
|total_quantity_sold|
+-------------------+
|             549473|
+-------------------+

47. Average Order Value:
+-------------------+
|average_order_value|
+-------------------+
|       19485.061938|
+-------------------+

--- Part 7: Aggregations Completed ---
Monthly Sales Sample:
+----+-----+-----------+
|year|month|total_sales|
+----+-----+-----------+
|2024|    1|26012137.99|
|2024|    2|24184417.63|
|2024|    3|26550683.95|
|2024|    4|26421814.55|
|2024|    5|26636293.47|
+----+-----+-----------+
only showing top 5 rows


In [8]:
from pyspark.sql.functions import col

# Extract DataFrames needed for advanced analytics
df_customers = dataframes["customers"]
df_orders = dataframes["orders"]
df_products = dataframes["products"]
df_order_details = dataframes["order_details"]
df_shipments = dataframes["shipments"]
df_payments = dataframes["payments"]
df_order_summary = dataframes["order_summary"] # Contains 'order_total' calculated in Part 4

# 57. Find the top 10 customers by sales[cite: 1]
top_10_customers = dataframes["customer_sales"].orderBy(col("total_sales").desc()).limit(10)
print("57. Top 10 Customers by Sales:")
top_10_customers.show()

# 58. Find the top 10 products by sales[cite: 1]
top_10_products = dataframes["product_sales"].orderBy(col("total_sales").desc()).limit(10)
print("58. Top 10 Products by Sales:")
top_10_products.show()

# 59. Find the top 5 categories by sales[cite: 1]
top_5_categories = dataframes["category_sales"].orderBy(col("total_sales").desc()).limit(5)
print("59. Top 5 Categories by Sales:")
top_5_categories.show()

# 60. Find customers who have never placed an order[cite: 1]
# A left_anti join returns rows from the left table that have no match in the right table
customers_no_orders = df_customers.join(df_orders, "customerid", "left_anti")
print("60. Customers with no orders:")
customers_no_orders.select("customerid", "full_name").show(5)

# 61. Find products that have never been ordered[cite: 1]
products_no_orders = df_products.join(df_order_details, "productid", "left_anti")
print("61. Products never ordered:")
products_no_orders.select("productid", "productname").show(5)

# 62. Find customers with more than 10 orders[cite: 1]
customers_over_10_orders = df_orders.groupBy("customerid").count().filter(col("count") > 10)
print("62. Customers with > 10 orders:")
customers_over_10_orders.show(5)

# 63. Find orders where the payment amount does not match the calculated order total[cite: 1]
# Group payments by orderid first in case there are multiple payments per order
payments_per_order = df_payments.groupBy("orderid").agg(spark_sum("amount").alias("total_paid"))
mismatched_payments = df_order_summary.join(payments_per_order, "orderid") \
    .filter(col("order_total") != col("total_paid"))
print("63. Orders with mismatched payment amounts:")
mismatched_payments.show(5)

# 64. Find orders that do not have a shipment[cite: 1]
orders_no_shipment = df_orders.join(df_shipments, "orderid", "left_anti")
print("64. Orders missing shipments:")
orders_no_shipment.select("orderid", "orderdate", "status").show(5)

# 65. Find shipments that do not have a matching order[cite: 1]
shipments_no_order = df_shipments.join(df_orders, "orderid", "left_anti")
print("65. Shipments missing matching orders:")
shipments_no_order.select("shipmentid", "orderid").show(5)

print("--- Part 8: Advanced Analytics Completed ---")

57. Top 10 Customers by Sales:
+----------+-----------+
|customerid|total_sales|
+----------+-----------+
|      6772|  368984.90|
|       245|  338329.93|
|      9895|  330803.47|
|      5446|  321600.74|
|      5688|  319630.49|
|      4971|  314924.38|
|      7013|  311641.81|
|      8502|  310361.58|
|      3556|  309252.08|
|       849|  307160.04|
+----------+-----------+

58. Top 10 Products by Sales:
+---------+--------------------+-----------+
|productid|         productname|total_sales|
+---------+--------------------+-----------+
|      582|Microsoft Printer...| 2128226.94|
|      794|   Canon Printer 794| 2037242.52|
|      650|       HP Laptop 650| 1954709.20|
|       71|    Dell Keyboard 71| 1915462.35|
|      911|Adidas Gaming Con...| 1858710.64|
|      232|   LG Headphones 232| 1857950.64|
|      191|      Sony Mouse 191| 1841616.40|
|      125| Logitech Laptop 125| 1830229.92|
|      335|      Sony Chair 335| 1826738.13|
|      350| Logitech Tablet 350| 1823336.58|
+--

In [9]:
from pyspark.sql.functions import col, sum as spark_sum, count, avg as spark_avg

# Extract necessary DataFrames
df_sales = dataframes["sales_dataset"] # Contains orderdate, year, month, quarter, total_amount[cite: 1]
df_orders = dataframes["orders"]

# 66. Calculate daily sales[cite: 1]
daily_sales = df_sales.groupBy("orderdate").agg(spark_sum("total_amount").alias("daily_sales")).orderBy("orderdate")
print("66. Daily Sales (First 5 days):")
daily_sales.show(5)

# 67. Calculate monthly sales[cite: 1]
monthly_sales = df_sales.groupBy("year", "month").agg(spark_sum("total_amount").alias("monthly_sales")).orderBy("year", "month")
print("67. Monthly Sales (First 5 months):")
monthly_sales.show(5)

# 68. Calculate quarterly sales[cite: 1]
quarterly_sales = df_sales.groupBy("year", "quarter").agg(spark_sum("total_amount").alias("quarterly_sales")).orderBy("year", "quarter")
print("68. Quarterly Sales:")
quarterly_sales.show(5)

# 69. Calculate yearly sales[cite: 1]
yearly_sales = df_sales.groupBy("year").agg(spark_sum("total_amount").alias("yearly_sales")).orderBy("year")
print("69. Yearly Sales:")
yearly_sales.show()

# 70. Find the month with the highest sales[cite: 1]
highest_sales_month = monthly_sales.orderBy(col("monthly_sales").desc()).limit(1)
print("70. Month with the highest sales:")
highest_sales_month.show()

# 71. Find the day with the highest number of orders[cite: 1]
orders_per_day = df_orders.groupBy("orderdate").agg(count("orderid").alias("total_orders"))
highest_orders_day = orders_per_day.orderBy(col("total_orders").desc()).limit(1)
print("71. Day with the highest number of orders:")
highest_orders_day.show()

# 72. Calculate the average number of orders per month[cite: 1]
orders_per_month = df_orders.groupBy("year", "month").agg(count("orderid").alias("orders_in_month"))
avg_orders_per_month = orders_per_month.select(spark_avg("orders_in_month").alias("avg_orders_per_month"))
print("72. Average number of orders per month:")
avg_orders_per_month.show()

print("--- Part 9: Date Analysis Completed ---")

66. Daily Sales (First 5 days):
+----------+-----------+
| orderdate|daily_sales|
+----------+-----------+
|2024-01-01|  797698.05|
|2024-01-02|  554259.51|
|2024-01-03|  956922.36|
|2024-01-04|  571969.71|
|2024-01-05| 1146885.03|
+----------+-----------+
only showing top 5 rows

67. Monthly Sales (First 5 months):
+----+-----+-------------+
|year|month|monthly_sales|
+----+-----+-------------+
|2024|    1|  26012137.99|
|2024|    2|  24184417.63|
|2024|    3|  26550683.95|
|2024|    4|  26421814.55|
|2024|    5|  26636293.47|
+----+-----+-------------+
only showing top 5 rows

68. Quarterly Sales:
+----+-------+---------------+
|year|quarter|quarterly_sales|
+----+-------+---------------+
|2024|      1|    76747239.57|
|2024|      2|    78744940.89|
|2024|      3|    78914874.64|
|2024|      4|    78678462.37|
|2025|      1|    76442662.12|
+----+-------+---------------+
only showing top 5 rows

69. Yearly Sales:
+----+------------+
|year|yearly_sales|
+----+------------+
|2024|31308

In [10]:
# ==========================================
# Part 11: Build Final Data Model (fact_sales)
# ==========================================

# Create fact_sales by selecting specific columns from the sales dataset[cite: 1]
# The sales_dataset from Part 6 already joins orders, order_details, and products.
fact_sales = dataframes["sales_dataset"].select(
    "orderid",
    "customerid",
    "productid",
    "categoryid",
    "orderdate",
    "quantity",
    "unitprice",
    "total_amount",
    "status"
)

# Add fact_sales and the required aggregated datasets to our dictionary for easy output mapping
output_datasets = {
    "customers": dataframes["customers"],
    "products": dataframes["products"],
    "categories": dataframes["categories"],
    "orders": dataframes["orders"],
    "order_details": dataframes["order_details"],
    "payments": dataframes["payments"],
    "shipments": dataframes["shipments"],
    "fact_sales": fact_sales,
    "customer_sales": dataframes["customer_sales"],
    "product_sales": dataframes["product_sales"],
    "category_sales": dataframes["category_sales"],
    "monthly_sales": dataframes["monthly_sales"]
}

# ==========================================
# Part 10: Output to S3 (Parquet + Snappy)
# ==========================================

processed_path = "s3://ecommerce-task28/processed/ecommerce/"

print("--- Starting Part 10 & 11: Writing Data to S3 ---")

for folder_name, df in output_datasets.items():
    output_location = f"{processed_path}{folder_name}/"
    print(f"Writing {folder_name} to {output_location}...")
    
    # Partition the orders dataset by year and month[cite: 1]
    if folder_name == "orders":
        df.write.mode("overwrite") \
            .partitionBy("year", "month") \
            .option("compression", "snappy") \
            .parquet(output_location)
    else:
        # Write standard datasets with snappy compression[cite: 1]
        df.write.mode("overwrite") \
            .option("compression", "snappy") \
            .parquet(output_location)

print("==================================================")
print("ETL Pipeline Complete. All files written to S3.")
print("==================================================")

--- Starting Part 10 & 11: Writing Data to S3 ---
Writing customers to s3://ecommerce-task28/processed/ecommerce/customers/...
Writing products to s3://ecommerce-task28/processed/ecommerce/products/...
Writing categories to s3://ecommerce-task28/processed/ecommerce/categories/...
Writing orders to s3://ecommerce-task28/processed/ecommerce/orders/...
Writing order_details to s3://ecommerce-task28/processed/ecommerce/order_details/...
Writing payments to s3://ecommerce-task28/processed/ecommerce/payments/...
Writing shipments to s3://ecommerce-task28/processed/ecommerce/shipments/...
Writing fact_sales to s3://ecommerce-task28/processed/ecommerce/fact_sales/...
Writing customer_sales to s3://ecommerce-task28/processed/ecommerce/customer_sales/...
Writing product_sales to s3://ecommerce-task28/processed/ecommerce/product_sales/...
Writing category_sales to s3://ecommerce-task28/processed/ecommerce/category_sales/...
Writing monthly_sales to s3://ecommerce-task28/processed/ecommerce/monthl